# Clustering Detection with Machine Learning

This notebook explores two ML approaches to detect scan clusters:
1. **Hidden Markov Model (HMM)** - Learns timing patterns unsupervised
2. **LSTM** - Deep learning sequence model for boundary detection

We'll build these step-by-step with explanations and visualizations.

## Part 0: Setup and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from hmmlearn import hmm  # Hidden Markov Models
from sklearn.mixture import GaussianMixture  # For GMM comparison
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Load processed scan data
parquet_path = Path('../data/processed/scans.parquet')
df = pd.read_parquet(parquet_path)

print(f"Loaded {len(df)} scan records from {df['experimentId'].nunique()} experiments")
print(f"\nFirst few rows:")
print(df.head())

## Part 1: Understanding the Problem

### What is clustering?

A **cluster** is a group of consecutive scans performed quickly (small deltaMs).
Clusters are separated by **pauses** (large deltaMs) when the user resets.

Example timing sequence:
```
Scan 0: deltaMs = null (first scan)
Scan 1: deltaMs = 300ms   <- cluster 1 interior
Scan 2: deltaMs = 250ms   <- cluster 1 interior  
Scan 3: deltaMs = 1500ms  <- PAUSE (reset)
Scan 4: deltaMs = 350ms   <- cluster 2 interior
Scan 5: deltaMs = 280ms   <- cluster 2 interior
```

The challenge: How do we automatically detect where the pauses are?

## Part 2: Visualize Timing Patterns

In [ ]:
# Pick a sample compliant experiment to visualize
compliant_exps = df[df['targetScanStyle'] == 'compliant']['experimentId'].unique()
sample_exp = compliant_exps[0]

sample_data = df[df['experimentId'] == sample_exp].sort_values('scanIndex').reset_index(drop=True)

print(f"Sample experiment: {sample_exp}")
print(f"User: {sample_data['userName'].iloc[0]}")
print(f"Target cluster size: {sample_data['targetClusterSize'].iloc[0]}")
print(f"Total scans: {len(sample_data)}")
print(f"\nTiming statistics (deltaMs):")
print(sample_data['deltaMs'].describe())

In [ ]:
# Visualize the timing sequence
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Timeline of deltaMs
deltas = sample_data['deltaMs'].fillna(0)
axes[0].bar(range(len(deltas)), deltas, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Scan Index')
axes[0].set_ylabel('Delta Ms (time to previous scan)')
axes[0].set_title(f'Scan Timing Pattern - Experiment {sample_exp}')
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribution of deltaMs
delta_data = sample_data['deltaMs'].dropna()
axes[1].hist(delta_data, bins=20, color='coral', alpha=0.7, edgecolor='black')
axes[1].axvline(delta_data.mean(), color='red', linestyle='--', label=f'Mean: {delta_data.mean():.0f}ms')
axes[1].axvline(delta_data.median(), color='green', linestyle='--', label=f'Median: {delta_data.median():.0f}ms')
axes[1].set_xlabel('Delta Ms')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Inter-Scan Times')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: Prepare Data for ML

### Feature Engineering

We need to prepare timing data for our ML models. The goal is to create sequences where we predict:
- **Class 0**: "Regular scan" (within a cluster)
- **Class 1**: "Pause/Boundary" (between clusters)

In [ ]:
def prepare_sequences_for_ml(df, sequence_length=10):
    """
    Convert scan data into sequences for ML models.
    
    For each experiment, we create sequences of deltaMs values.
    This allows models to learn patterns from neighboring scans.
    
    Args:
        df: Processed scan dataframe
        sequence_length: Number of previous scans to consider (context window)
    
    Returns:
        X: Feature sequences (2D array of deltaMs values)
        exp_ids: Which experiment each sequence came from
    """
    sequences = []
    experiment_ids = []
    
    # For each experiment
    for exp_id in df['experimentId'].unique():
        exp_data = df[df['experimentId'] == exp_id].sort_values('scanIndex')
        deltas = exp_data['deltaMs'].fillna(0).values  # Fill first scan (null) with 0
        
        # Skip experiments with too few scans
        if len(deltas) < sequence_length:
            continue
        
        # Create sliding windows of timing data
        for i in range(len(deltas) - sequence_length + 1):
            window = deltas[i:i + sequence_length]
            sequences.append(window)
            experiment_ids.append(exp_id)
    
    X = np.array(sequences).reshape(-1, sequence_length, 1)  # Shape: (n_sequences, sequence_length, 1)
    return X, np.array(experiment_ids), sequences

# Prepare sequences
X_sequences, exp_ids, raw_sequences = prepare_sequences_for_ml(df, sequence_length=10)
print(f"Created {len(X_sequences)} sequences from {len(np.unique(exp_ids))} experiments")
print(f"Sequence shape: {X_sequences.shape}")
print(f"  - {X_sequences.shape[0]} sequences")
print(f"  - {X_sequences.shape[1]} time steps per sequence")
print(f"  - {X_sequences.shape[2]} features per time step")

## Part 4: Hidden Markov Model (HMM)

### What is an HMM?

An HMM assumes there are **hidden states** that generate the observed data.

In our case:
- **State 1**: "Cluster interior" - fast scans, small deltaMs
- **State 2**: "Pause/Reset" - slow scans, large deltaMs

The HMM learns:
1. What timing patterns (deltaMs) are typical for each state
2. How often we transition between states
3. Which state most likely generated each observed scan

**Advantage**: No labeled data needed! It learns from timing patterns alone.

In [ ]:
# Prepare data for HMM (flatten from sequences)
# HMM expects a 1D or 2D array of observations
X_hmm = X_sequences.reshape(-1, 1)  # Flatten to single column of deltaMs values

print(f"Preparing HMM with {len(X_hmm)} timing observations")
print(f"Data range: {X_hmm.min():.0f}ms to {X_hmm.max():.0f}ms")

In [ ]:
# Train HMM with 2 states (cluster interior vs pause)
print("Training HMM with 2 states...")

model_hmm = hmm.GaussianHMM(n_components=2, covariance_type="full", n_iter=1000)
model_hmm.fit(X_hmm)

print("HMM trained!")
print(f"\nHMM Results:")
print(f"State 0 mean deltaMs: {model_hmm.means_[0][0]:.1f}ms")
print(f"State 1 mean deltaMs: {model_hmm.means_[1][0]:.1f}ms")
print(f"\nState covariances:")
print(f"State 0 variance: {model_hmm.covars_[0][0, 0]:.1f}")
print(f"State 1 variance: {model_hmm.covars_[1][0, 0]:.1f}")
print(f"\nTransition probabilities (how often we switch states):")
print(model_hmm.transmat_)

In [ ]:
# Get state predictions for our sample experiment
sample_deltas = sample_data['deltaMs'].fillna(0).values.reshape(-1, 1)
sample_states = model_hmm.predict(sample_deltas)

# Visualize HMM results
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Timing with predicted states
colors = ['steelblue' if s == 0 else 'coral' for s in sample_states]
axes[0].bar(range(len(sample_deltas)), sample_deltas.flatten(), color=colors, alpha=0.7)
axes[0].set_xlabel('Scan Index')
axes[0].set_ylabel('Delta Ms')
axes[0].set_title(f'HMM Clustering Detection - {sample_exp}')
axes[0].legend(['State 0 (Cluster)', 'State 1 (Pause)'])
axes[0].grid(True, alpha=0.3)

# Plot 2: State sequence
axes[1].scatter(range(len(sample_states)), sample_states, s=100, alpha=0.6, c=sample_states, cmap='coolwarm')
axes[1].plot(range(len(sample_states)), sample_states, alpha=0.3)
axes[1].set_xlabel('Scan Index')
axes[1].set_ylabel('Predicted State')
axes[1].set_title('State Predictions Over Time')
axes[1].set_ylim(-0.5, 1.5)
axes[1].set_yticks([0, 1])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print interpretation
# Determine which state is "pause" (higher mean)
pause_state = 1 if model_hmm.means_[1][0] > model_hmm.means_[0][0] else 0
cluster_state = 1 - pause_state

print(f"\nInterpretation:")
print(f"State {cluster_state} = Cluster Interior (mean {model_hmm.means_[cluster_state][0]:.1f}ms)")
print(f"State {pause_state} = Pause/Boundary (mean {model_hmm.means_[pause_state][0]:.1f}ms)")
print(f"\nDetected states in sample: {sample_states}")

## Part 5: Extract Clusters from HMM Predictions

In [ ]:
def extract_clusters_from_states(states, pause_state=1):
    """
    Convert state predictions into cluster assignments.
    
    A new cluster starts after a pause state.
    """
    clusters = []
    current_cluster = 0
    
    for i, state in enumerate(states):
        if state == pause_state:
            # Next scan will be in a new cluster
            current_cluster += 1
        clusters.append(current_cluster)
    
    return np.array(clusters)

# Extract clusters
pause_state = 1 if model_hmm.means_[1][0] > model_hmm.means_[0][0] else 0
sample_clusters = extract_clusters_from_states(sample_states, pause_state=pause_state)

# Analyze detected clusters
print(f"Detected {sample_clusters.max() + 1} clusters")
print(f"\nCluster sizes:")
for c_id in range(sample_clusters.max() + 1):
    c_size = np.sum(sample_clusters == c_id)
    print(f"  Cluster {c_id}: {c_size} scans")

print(f"\nTarget cluster size: {sample_data['targetClusterSize'].iloc[0]}")
print(f"Expected total clusters: ~{len(sample_data) / sample_data['targetClusterSize'].iloc[0]:.0f}")

## Part 6: LSTM for Boundary Detection

### What is LSTM?

LSTM (Long Short-Term Memory) is a deep learning model that:
- Processes sequences and learns temporal patterns
- Remembers patterns from previous scans
- Predicts whether each scan is a boundary or cluster-interior

**Advantage**: Can learn more complex patterns than HMM, leverages context.

We'll train it using HMM predictions as weak labels (not perfect, but good enough to start).

In [ ]:
# Create labels for LSTM training
# A scan is a "boundary" if the next deltaMs is large (pause)

def create_lstm_labels(df, pause_threshold_percentile=75):
    """
    Create labels for boundary detection.
    
    A scan is labeled as 1 (boundary) if the deltaMs to the NEXT scan is large.
    We use a percentile threshold to define "large".
    """
    X_list = []
    y_list = []
    
    for exp_id in df['experimentId'].unique():
        exp_data = df[df['experimentId'] == exp_id].sort_values('scanIndex').reset_index(drop=True)
        deltas = exp_data['deltaMs'].fillna(0).values
        
        if len(deltas) < 10:
            continue
        
        # Calculate pause threshold
        pause_threshold = np.percentile(deltas, pause_threshold_percentile)
        
        # Create labels: 1 if next scan is a pause, 0 otherwise
        labels = []
        for i in range(len(deltas) - 1):
            label = 1 if deltas[i + 1] > pause_threshold else 0
            labels.append(label)
        # Last scan is ambiguous, skip it
        
        # Create sequences
        for i in range(len(deltas) - 10):
            window = deltas[i:i + 10]
            X_list.append(window)
            y_list.append(labels[i + 9])  # Label for the last position
    
    X = np.array(X_list).reshape(-1, 10, 1)
    y = np.array(y_list)
    
    return X, y

print("Creating LSTM training data...")
X_lstm, y_lstm = create_lstm_labels(df, pause_threshold_percentile=75)

print(f"LSTM dataset created:")
print(f"  X shape: {X_lstm.shape}")
print(f"  y shape: {y_lstm.shape}")
print(f"  Class distribution: {np.sum(y_lstm == 0)} cluster, {np.sum(y_lstm == 1)} boundary")

In [ ]:
# Split data into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm, y_lstm, test_size=0.2, random_state=42, stratify=y_lstm
)

print(f"Train set: {X_train.shape[0]} sequences")
print(f"Test set: {X_test.shape[0]} sequences")

In [ ]:
# Build LSTM model
print("Building LSTM model...")

model_lstm = Sequential([
    # LSTM layer: learns patterns in sequences
    LSTM(64, activation='relu', input_shape=(10, 1), return_sequences=True),
    Dropout(0.2),  # Prevent overfitting
    
    # Second LSTM layer for deeper learning
    LSTM(32, activation='relu'),
    Dropout(0.2),
    
    # Dense layers for classification
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # Output: probability of boundary (0 or 1)
])

# Compile model
model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model_lstm.summary())

In [ ]:
# Train LSTM
print("Training LSTM...")

history = model_lstm.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model_lstm.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Part 7: Compare HMM vs LSTM

Let's test both models on a held-out sample experiment.

In [ ]:
# Pick another sample experiment
test_exp = compliant_exps[3]  # Different from our original sample
test_data = df[df['experimentId'] == test_exp].sort_values('scanIndex').reset_index(drop=True)
test_deltas = test_data['deltaMs'].fillna(0).values

print(f"Test experiment: {test_exp}")
print(f"Scans: {len(test_data)}, Target cluster size: {test_data['targetClusterSize'].iloc[0]}")

In [ ]:
# HMM predictions
test_hmm_states = model_hmm.predict(test_deltas.reshape(-1, 1))
test_hmm_clusters = extract_clusters_from_states(test_hmm_states, pause_state=pause_state)

# LSTM predictions (prepare as sequence)
# Pad to 10-length sequences
test_lstm_input = test_deltas[:10].reshape(1, 10, 1)  # Take first 10 scans
test_lstm_probs = model_lstm.predict(test_lstm_input, verbose=0)[0]

print(f"LSTM boundary probability: {test_lstm_probs:.3f}")
print(f"(>0.5 suggests a boundary, <0.5 suggests cluster continues)")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# HMM
colors_hmm = ['steelblue' if c % 2 == 0 else 'coral' for c in test_hmm_clusters]
axes[0].bar(range(len(test_deltas)), test_deltas, color=colors_hmm, alpha=0.7)
axes[0].set_ylabel('Delta Ms')
axes[0].set_title(f'HMM Clustering - {test_exp}')
axes[0].grid(True, alpha=0.3)

# Show cluster boundaries
cluster_changes = np.where(np.diff(test_hmm_clusters) != 0)[0] + 1
for change_idx in cluster_changes:
    axes[0].axvline(change_idx, color='red', linestyle='--', alpha=0.5, linewidth=2)

# LSTM predictions on a rolling basis
lstm_preds = []
for i in range(len(test_deltas) - 9):
    window = test_deltas[i:i+10].reshape(1, 10, 1)
    pred = model_lstm.predict(window, verbose=0)[0][0]
    lstm_preds.append(pred)

# Plot only where we have predictions
lstm_range = range(9, len(test_deltas))  # Start from position 9
axes[1].plot(lstm_range, lstm_preds, marker='o', markersize=6, linewidth=2, color='green')
axes[1].axhline(0.5, color='red', linestyle='--', label='Decision boundary')
axes[1].set_xlabel('Scan Index')
axes[1].set_ylabel('Boundary Probability')
axes[1].set_title(f'LSTM Boundary Detection - {test_exp}')
axes[1].set_ylim(-0.1, 1.1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

### HMM (Hidden Markov Model)
✅ **Pros:**
- Unsupervised (no labels needed)
- Interpretable (learns state means and variances)
- Fast to train
- Good for initial exploration

❌ **Cons:**
- Assumes 2 distinct distributions (may fail if they overlap)
- Simpler, may miss complex patterns

### LSTM (Deep Learning)
✅ **Pros:**
- Learns complex temporal patterns
- Can capture context from surrounding scans
- Provides confidence scores
- More flexible

❌ **Cons:**
- Requires labeled training data
- Harder to interpret
- More computationally expensive

### Recommendation

**Use HMM to generate weak labels** → **Train LSTM for production**

1. Use HMM for fast initial clustering on all data
2. Use HMM predictions as weak labels for LSTM
3. Fine-tune LSTM with any manually labeled data if available
4. Deploy LSTM for inference (boundary detection for compliance classification)